# Hafta 5 · Tek Kübit Kapıları II: Faz ve Döndürme Kapıları, Devre Optimizasyonu
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

4\. haftada X, Y, Z ve H kapılarını Bloch küresinde tek tek incelemiştik. Bu hafta aynı yöntemle **faz kapılarını** (S, T, P), **döndürme kapılarını** (Rx, Ry, Rz) ve hepsini kapsayan **genel U kapısını** inceliyoruz. Ardından istediğimiz her tek kübit durumunu hazırlayan bir `prepare()` fonksiyonu yazıyor, son olarak devreleri **ölçüp (derinlik, kapı sayısı)** Qiskit'in derleyicisi `transpile` ile optimize ediyoruz.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar (Bloch çizimi, iz ile) | 3 dk |
| A | Faz kapıları: S, S†, T, T†, P(λ) | 7 dk |
| B | Döndürme kapıları: Rx, Ry, Rz; θ taraması | 10 dk |
| C | (Opsiyonel) θ kaydırıcılı etkileşimli Bloch | 2 dk |
| D | Genel U(θ, φ, λ) kapısı ve Euler ayrıştırması | 7 dk |
| E | Kapı birleştirme ve değişmezlik | 5 dk |
| F | Durum hazırlama: `prepare(α, β)` ve "hedef duruma ulaş" bulmacaları | 8 dk |
| G | Devre metrikleri ve `transpile` (optimization_level 0–3) | 8 dk |
| H | Alıştırmalar (8 adet) | ödev |

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator, state_fidelity
from qiskit.synthesis import OneQubitEulerDecomposer

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY, PALE = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6", "#AEB8C8"
rng = np.random.default_rng(2026)
aer = AerSimulator(seed_simulator=11)

def state_to_bloch(amps):
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def _sphere(ax):
    ax.set_box_aspect((1,1,1), zoom=1.3); ax.computed_zorder = False
    u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
    ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)),
                    color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
    t = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8); ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
    for d in [(1,0,0), (0,1,0), (0,0,1)]: ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
    for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"), ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
        ax.text(*p, s, ha="center", va="center", fontsize=9.5, color=NAVY)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1); ax.view_init(elev=18, azim=30); ax.set_axis_off()

def _vec(ax, v, color):
    ax.plot([0, v[0]], [0, v[1]], [0, v[2]], color=color, lw=3); ax.scatter([v[0]], [v[1]], [v[2]], color=color, s=60, depthshade=False)

def plot_bloch(amps_list, titles=None, starts=None, trails=None):
    """Kübit durumlarını yan yana Bloch küresinde çizer (4. haftadaki fonksiyonun izli sürümü).
    starts: her panel için başlangıç durumu (soluk çizilir); trails: (k,3) Bloch noktaları (turuncu iz)."""
    if np.ndim(amps_list) == 1: amps_list = [amps_list]
    k = len(amps_list); fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, amps in enumerate(amps_list):
        ax = fig.add_subplot(1, k, i+1, projection="3d"); _sphere(ax)
        if trails is not None and trails[i] is not None:
            tr = np.asarray(trails[i]); ax.plot(tr[:,0], tr[:,1], tr[:,2], color=ORANGE, lw=2, ls=":")
        if starts is not None and starts[i] is not None: _vec(ax, state_to_bloch(starts[i]), PALE)
        _vec(ax, state_to_bloch(amps), BLUE)
        if titles: ax.set_title(titles[i], fontsize=11, color=NAVY)
    plt.show()

def gate_path(U_of_t, start, t_end, n=50):
    """Parametreli kapı U(t) için t: 0 → t_end boyunca Bloch yolunu (iz) üretir."""
    return np.array([state_to_bloch(U_of_t(t) @ start) for t in np.linspace(0, t_end, n)])

ket0 = np.array([1, 0], dtype=complex); ket1 = np.array([0, 1], dtype=complex)
plus = np.array([1, 1], dtype=complex)/np.sqrt(2)
H = np.array([[1, 1], [1, -1]], dtype=complex)/np.sqrt(2)
X = np.array([[0, 1], [1, 0]], dtype=complex); Z = np.diag([1, -1]).astype(complex)
print("hazır")

---
## A · Faz kapıları: S, S†, T, T†, P(λ)
Faz kapıları |0⟩ genliğine dokunmaz, |1⟩ genliğini bir **faz çarpanı** e^(iλ) ile çarpar:

$$P(\lambda)=\begin{pmatrix}1&0\\0&e^{i\lambda}\end{pmatrix}\qquad S=P(\pi/2),\; T=P(\pi/4),\; S^\dagger=P(-\pi/2),\; T^\dagger=P(-\pi/4),\; Z=P(\pi)$$

| Kapı | λ | Bloch'ta etkisi | Qiskit |
|---|---|---|---|
| T | π/4 | z etrafında +45° | `qc.t(0)` |
| S | π/2 | z etrafında +90° | `qc.s(0)` |
| Z | π | z etrafında 180° | `qc.z(0)` |
| S† | −π/2 | z etrafında −90° | `qc.sdg(0)` |
| T† | −π/4 | z etrafında −45° | `qc.tdg(0)` |
| P(λ) | λ | z etrafında λ | `qc.p(lam, 0)` |

**Önemli:** Faz kapıları ölçüm olasılıklarını **değiştirmez** (|e^(iλ)| = 1). Etkilerini görmek için kübiti önce ekvatora (ör. |+⟩) koymak gerekir.

In [ ]:
def P(lam): return np.array([[1, 0], [0, np.exp(1j*lam)]])
S, Sdg, T, Tdg = P(np.pi/2), P(-np.pi/2), P(np.pi/4), P(-np.pi/4)

# Qiskit ile karşılaştırma: her kapıyı tek kapılık devreye koyup matrisini alalım
for name, M in [("s", S), ("sdg", Sdg), ("t", T), ("tdg", Tdg)]:
    qc = QuantumCircuit(1); getattr(qc, name)(0)
    print(f"{name:4s} Qiskit ile aynı mı? {np.allclose(Operator(qc).data, M)}")

# Faz kapıları olasılığı değiştirmez:
print("\nS|+⟩ =", S @ plus, "  olasılıklar:", abs(S @ plus)**2)

In [ ]:
# |+⟩ durumuna S, S†, T, T† uygulayıp Bloch'ta izleriyle çizelim
gates = [("S", np.pi/2), ("S†", -np.pi/2), ("T", np.pi/4), ("T†", -np.pi/4)]
plot_bloch([P(l) @ plus for _, l in gates], [f"{n}|+⟩" for n, _ in gates],
           starts=[plus]*4, trails=[gate_path(P, plus, l) for _, l in gates])

In [ ]:
# Film şeridi: P(λ)|+⟩, λ büyüdükçe nokta ekvatorda döner
lams = [np.pi/4, np.pi/2, np.pi, 3*np.pi/2]
plot_bloch([P(l) @ plus for l in lams], ["P(π/4)", "P(π/2)", "P(π)", "P(3π/2)"],
           starts=[plus]*4, trails=[gate_path(P, plus, l) for l in lams])

### Faz kapılarının cebiri: açılar toplanır
P(a)·P(b) = P(a + b). Buradan T² = S, S² = Z, Z² = I, T⁸ = I çıkar. Kodla doğrulayalım:

In [ ]:
def same(A, B): return np.allclose(A, B)
tablo = [("T·T = S", same(T @ T, S)), ("S·S = Z", same(S @ S, Z)), ("Z·Z = I", same(Z @ Z, np.eye(2))),
         ("T⁸ = I", same(np.linalg.matrix_power(T, 8), np.eye(2))), ("S·S† = I", same(S @ Sdg, np.eye(2))),
         ("T† = T'nin eşlenik transpozu", same(Tdg, T.conj().T)), ("P(0.3)·P(0.5) = P(0.8)", same(P(0.3) @ P(0.5), P(0.8)))]
for k, v in tablo: print(f"{k:32s} {v}")

---
## B · Döndürme kapıları: Rx(θ), Ry(θ), Rz(θ)
Döndürme kapıları Bloch vektörünü seçilen eksen etrafında **θ kadar** döndürür. Parametre alan kapılardır; kuantum makine öğrenmesinde (12–15. haftalar) "öğrenilen ağırlık" rolünü bunlar oynayacak.

$$R_x(\theta)=\begin{pmatrix}\cos\frac{\theta}{2}&-i\sin\frac{\theta}{2}\\-i\sin\frac{\theta}{2}&\cos\frac{\theta}{2}\end{pmatrix}\quad
R_y(\theta)=\begin{pmatrix}\cos\frac{\theta}{2}&-\sin\frac{\theta}{2}\\\sin\frac{\theta}{2}&\cos\frac{\theta}{2}\end{pmatrix}\quad
R_z(\theta)=\begin{pmatrix}e^{-i\theta/2}&0\\0&e^{i\theta/2}\end{pmatrix}$$

**Neden θ/2?** Bloch küresindeki açı, genlik vektörleri arasındaki açının **iki katıdır**. |0⟩ = [1, 0] ile |1⟩ = [0, 1] genlik düzleminde 90° ayrıktır ama Bloch'ta 180° (kuzey ve güney kutbu). Bloch'ta θ kadar dönmek için genlikte θ/2 kadar dönmek gerekir.

In [ ]:
def Rx(t): return np.array([[np.cos(t/2), -1j*np.sin(t/2)], [-1j*np.sin(t/2), np.cos(t/2)]])
def Ry(t): return np.array([[np.cos(t/2), -np.sin(t/2)], [np.sin(t/2), np.cos(t/2)]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2), 0], [0, np.exp(1j*t/2)]])

th = 0.73
for name, f in [("rx", Rx), ("ry", Ry), ("rz", Rz)]:
    qc = QuantumCircuit(1); getattr(qc, name)(th, 0)
    print(f"{name}(0.73) Qiskit ile aynı mı? {np.allclose(Operator(qc).data, f(th))}")

print("\nRy(θ)|0⟩ = [cos(θ/2), sin(θ/2)]  ->  θ = π/2 için", Ry(np.pi/2) @ ket0, "= |+⟩")
print("Rx(π) =\n", Rx(np.pi), "\n= −i·X  (X ile aynı kapı, sadece global faz farkı)")

In [ ]:
# Film şeritleri: |0⟩'a Rx(θ) ve Ry(θ)
thetas, names = [np.pi/4, np.pi/2, np.pi, 3*np.pi/2], ["π/4", "π/2", "π", "3π/2"]
plot_bloch([Rx(t) @ ket0 for t in thetas], [f"Rx({n})|0⟩" for n in names], starts=[ket0]*4, trails=[gate_path(Rx, ket0, t) for t in thetas])
plot_bloch([Ry(t) @ ket0 for t in thetas], [f"Ry({n})|0⟩" for n in names], starts=[ket0]*4, trails=[gate_path(Ry, ket0, t) for t in thetas])
# Rz |0⟩'ı değiştirmez (kutup eksen üzerinde!). Eğik bir durumdan başlayalım:
psi = Ry(np.pi/3) @ ket0
plot_bloch([Rz(t) @ psi for t in thetas], [f"Rz({n})|ψ⟩" for n in names], starts=[psi]*4, trails=[gate_path(Rz, psi, t) for t in thetas])

### θ taraması: P(1) = sin²(θ/2)
Ry(θ)|0⟩ = [cos(θ/2), sin(θ/2)] olduğundan P(1) = sin²(θ/2). Formülü Aer simülatöründe 2000 shot ile doğrulayalım.

In [ ]:
grid = np.linspace(0, 2*np.pi, 17)
est = []
for t in grid:
    qc = QuantumCircuit(1, 1); qc.ry(t, 0); qc.measure(0, 0)
    counts = aer.run(transpile(qc, aer), shots=2000).result().get_counts()
    est.append(counts.get("1", 0) / 2000)

tt = np.linspace(0, 2*np.pi, 300)
plt.figure(figsize=(8, 3.8))
plt.plot(tt, np.sin(tt/2)**2, color=BLUE, lw=2.5, label="teori: sin²(θ/2)")
plt.plot(grid, est, "o", color=ORANGE, label="Aer, 2000 shot")
plt.xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi], ["0", "π/2", "π", "3π/2", "2π"])
plt.xlabel("θ"); plt.ylabel("P(1)"); plt.title("Ry(θ)|0⟩ ölçümü", color=NAVY); plt.legend(frameon=False); plt.grid(alpha=0.25); plt.show()

In [ ]:
# Rx ve Rz için aynı tarama: Rx aynı olasılıkları verir (farkı fazdadır), Rz hiç değiştirmez
tablo = []
for t in [0, np.pi/3, np.pi/2, np.pi]:
    row = []
    for name in ["rx", "ry", "rz"]:
        qc = QuantumCircuit(1); getattr(qc, name)(t, 0)
        row.append(Statevector(qc).probabilities()[1])
    tablo.append((t, *row))
print("  θ      P1(Rx)  P1(Ry)  P1(Rz)")
for r in tablo: print(f"{r[0]:5.3f}   {r[1]:.3f}   {r[2]:.3f}   {r[3]:.3f}")

### Rz ile P arasındaki fark: sadece global faz
$R_z(\lambda) = e^{-i\lambda/2}\,P(\lambda)$. İki matris farklıdır ama **aynı kuantum işlemidir** (2. haftadaki global faz). Qiskit'te `Operator.equiv` global fazı yok sayarak karşılaştırır.

In [ ]:
lam = np.pi/2
print("Rz(π/2) =\n", Rz(lam)); print("P(π/2)  =\n", P(lam))
print("Birebir eşit mi?", np.allclose(Rz(lam), P(lam)))
print("Rz = e^(−iλ/2)·P ?", np.allclose(Rz(lam), np.exp(-1j*lam/2) * P(lam)))
qa = QuantumCircuit(1); qa.rz(lam, 0); qb = QuantumCircuit(1); qb.p(lam, 0)
print("Operator.equiv (global faz hariç):", Operator(qa).equiv(Operator(qb)))
print("Bloch noktaları:", state_to_bloch(Rz(lam) @ plus), state_to_bloch(P(lam) @ plus))

---
## C · (Opsiyonel) θ kaydırıcılı etkileşimli Bloch
Aşağıdaki hücre önce **tek bir statik çizim** yapar. Colab'da `ipywidgets` yüklü olduğu için ayrıca bir kaydırıcı (slider) da açılır; kaydırıcıyı oynatarak kapının küre üzerinde nasıl döndüğünü izleyin. (`ipywidgets` bulunmayan ortamlarda hücre hata vermez, sadece statik çizimi gösterir.)

In [ ]:
def bloch_theta(theta=1.0, kapi="ry"):
    """Seçilen döndürme kapısını |0⟩ (Rz için eğik bir durum) üzerine θ açısıyla uygular ve iziyle çizer."""
    f = {"rx": Rx, "ry": Ry, "rz": Rz}[kapi]
    start = Ry(np.pi/3) @ ket0 if kapi == "rz" else ket0
    out = f(theta) @ start
    plot_bloch([out], [f"{kapi}({theta:.2f})  P(1) = {abs(out[1])**2:.3f}"], starts=[start], trails=[gate_path(f, start, theta)])

bloch_theta(2.0, "ry")     # statik çağrı

try:
    from ipywidgets import interact, FloatSlider
    interact(bloch_theta, theta=FloatSlider(min=0, max=2*np.pi, step=0.05, value=1.0), kapi=["rx", "ry", "rz"])
except ImportError:
    print("ipywidgets bulunamadı; Colab'da kaydırıcı otomatik açılır.")

---
## D · Genel U(θ, φ, λ) kapısı ve Euler ayrıştırması
Qiskit'in genel tek kübit kapısı:
$$U(\theta,\varphi,\lambda)=\begin{pmatrix}\cos\frac{\theta}{2} & -e^{i\lambda}\sin\frac{\theta}{2}\\ e^{i\varphi}\sin\frac{\theta}{2} & e^{i(\varphi+\lambda)}\cos\frac{\theta}{2}\end{pmatrix}$$

**Her** tek kübit kapısı (global faz dışında) bir U'dur. Yazılım benzetmesi: U, bütün tek kübit kapılarının türediği "temel sınıf"tır; X, H, S… sadece parametreleri sabitlenmiş örnekleridir.

**Euler ayrıştırması:** Her tek kübit kapısı için U = e^(iα) · Rz(φ) · Ry(θ) · Rz(λ). Yani **3 açı** her tek kübit kapısını tarif etmeye yeter.

In [ ]:
def U3(theta, phi, lam):
    c, s = np.cos(theta/2), np.sin(theta/2)
    return np.array([[c, -np.exp(1j*lam)*s], [np.exp(1j*phi)*s, np.exp(1j*(phi+lam))*c]])

qc = QuantumCircuit(1); qc.u(0.4, 1.1, -0.7, 0)
print("U3 Qiskit ile aynı mı?", np.allclose(Operator(qc).data, U3(0.4, 1.1, -0.7)))

dec = OneQubitEulerDecomposer("U")           # hedef: tek U kapısı
Y = np.array([[0, -1j], [1j, 0]])
print("\nKapı   θ        φ        λ       global faz")
for name, M in [("X", X), ("Y", Y), ("Z", Z), ("H", H), ("S", S), ("T", T)]:
    th_, ph_, la_, gp = dec.angles_and_phase(M)
    print(f"{name:4s} {th_:+.4f}  {ph_:+.4f}  {la_:+.4f}  {gp:+.4f}")

In [ ]:
# Euler ZYZ: bir kapıyı Rz·Ry·Rz devresine çevirelim
zyz = OneQubitEulerDecomposer("ZYZ")
for name, M in [("H", H), ("S", S), ("X", X)]:
    circ = zyz(M)
    print(f"{name}: {[(i.operation.name, round(float(i.operation.params[0]), 4)) for i in circ.data]}, global faz = {circ.global_phase:.4f}")
    print("   eşdeğer mi?", Operator(circ).equiv(Operator(M)))
zyz(H).draw("mpl")

In [ ]:
# Rastgele bir üniter matris üretip ayrıştıralım (her 2x2 üniter matris 3 açıyla yazılabilir)
from qiskit.quantum_info import random_unitary
Urand = random_unitary(2, seed=5).data
th_, ph_, la_ = dec.angles(Urand)
qc = QuantumCircuit(1); qc.u(th_, ph_, la_, 0)
print(f"θ={th_:.4f}, φ={ph_:.4f}, λ={la_:.4f}")
print("U(θ,φ,λ) rastgele matrisle eşdeğer mi?", Operator(qc).equiv(Operator(Urand)))

---
## E · Kapı birleştirme ve değişmezlik
- **Aynı eksen:** açılar toplanır → Rz(a)·Rz(b) = Rz(a+b), Rx(a)·Rx(b) = Rx(a+b).
- **Farklı eksen:** sıra önemlidir → Rx(a)·Ry(b) ≠ Ry(b)·Rx(a). Bloch'ta iki farklı yol, iki farklı son nokta.

In [ ]:
a, b = 0.4, 1.3
print("Rz(a)Rz(b) = Rz(a+b) ?", np.allclose(Rz(a) @ Rz(b), Rz(a + b)))
print("Rx(a)Rx(b) = Rx(a+b) ?", np.allclose(Rx(a) @ Rx(b), Rx(a + b)))

A = Ry(np.pi/2) @ Rx(np.pi/2)       # önce Rx, sonra Ry
B = Rx(np.pi/2) @ Ry(np.pi/2)       # önce Ry, sonra Rx
print("\nRy·Rx = Rx·Ry ?", np.allclose(A, B))
print("önce Rx sonra Ry -> Bloch", state_to_bloch(A @ ket0).round(3), " (|−i⟩)")
print("önce Ry sonra Rx -> Bloch", state_to_bloch(B @ ket0).round(3), " (|+⟩)")

p1 = gate_path(Rx, ket0, np.pi/2); p2 = gate_path(Ry, Rx(np.pi/2) @ ket0, np.pi/2)
q1 = gate_path(Ry, ket0, np.pi/2); q2 = gate_path(Rx, Ry(np.pi/2) @ ket0, np.pi/2)
plot_bloch([A @ ket0, B @ ket0], ["Rx(π/2) → Ry(π/2)", "Ry(π/2) → Rx(π/2)"], starts=[ket0, ket0], trails=[np.vstack([p1, p2]), np.vstack([q1, q2])])

---
## F · Durum hazırlama: `prepare(α, β)`
Hedef durum [α, β] verildiğinde |0⟩'dan ona giden devre:
1. **Enlem:** θ = 2·arccos|α| → `ry(θ)` genlik büyüklüklerini ayarlar: [|α|, |β|]
2. **Boylam:** φ = arg β − arg α → `p(φ)` göreli fazı ayarlar
3. Global faz (arg α) önemsizdir, atlanır.

Doğrulama ölçütü: **fidelity** F = |⟨hedef|ψ⟩|² (1 = aynı durum). Qiskit: `state_fidelity`.

In [ ]:
def prepare(alpha, beta):
    """|0⟩'dan [alpha, beta] durumunu hazırlayan 1 kübitlik devre döndürür."""
    v = np.array([alpha, beta], dtype=complex); v = v / np.linalg.norm(v)
    theta = 2*np.arccos(np.clip(abs(v[0]), 0, 1))
    phi = np.angle(v[1]) - np.angle(v[0])
    qc = QuantumCircuit(1); qc.ry(theta, 0); qc.p(phi, 0)
    return qc

target = np.array([0.6, 0.8j])
qc = prepare(*target)
print("fidelity:", state_fidelity(Statevector(qc), Statevector(target)))
qc.draw("mpl")

In [ ]:
# 200 rastgele hedefte test (birim test yaklaşımı)
worst = 1.0
for _ in range(200):
    v = rng.normal(size=2) + 1j*rng.normal(size=2); v /= np.linalg.norm(v)
    worst = min(worst, state_fidelity(Statevector(prepare(*v)), Statevector(v)))
print("200 rastgele hedefte en kötü fidelity:", worst)
assert worst >= 0.999

### "Hedef duruma ulaş" bulmacaları
Kural: |0⟩'dan başla, **sadece izin verilen kapıları** kullan, fidelity ≥ 0.999 olsun. Aşağıdaki kontrol fonksiyonu hem kapı setini hem de fidelity'yi denetler.

| Bulmaca | Hedef | İzin verilen kapılar |
|---|---|---|
| B1 | \|−i⟩ = [1, −i]/√2 | {H, S} |
| B2 | \|1⟩ | {H, T} (X yasak!) |
| B3 | [cos(π/8), e^(iπ/4)·sin(π/8)] | {H, T} + tek bir Ry |
| B4 | [0.6, −0.8] | {Ry, Z} |

In [ ]:
def check_puzzle(qc, target, allowed):
    used = set(qc.count_ops())
    assert used <= set(allowed), f"izin verilmeyen kapı: {used - set(allowed)}"
    F = state_fidelity(Statevector(qc), Statevector(np.asarray(target, dtype=complex)))
    assert F >= 0.999, f"fidelity = {F:.4f}"
    return F

# Örnek çözümler (derste birlikte)
b1 = QuantumCircuit(1); b1.h(0); b1.s(0); b1.s(0); b1.s(0)                 # S³ = S†  → |+⟩'yı −90° döndürür
b2 = QuantumCircuit(1); b2.h(0); [b2.t(0) for _ in range(4)]; b2.h(0)      # H·T⁴·H = H·Z·H = X
b3 = QuantumCircuit(1); b3.ry(np.pi/4, 0); b3.t(0)                         # enlem π/4, boylam π/4
b4 = QuantumCircuit(1); b4.ry(2*np.arccos(0.6), 0); b4.z(0)                # [0.6, 0.8] → Z → [0.6, −0.8]
for name, qc, tgt, al in [("B1", b1, np.array([1, -1j])/np.sqrt(2), ["h", "s"]), ("B2", b2, [0, 1], ["h", "t"]),
                          ("B3", b3, [np.cos(np.pi/8), np.exp(1j*np.pi/4)*np.sin(np.pi/8)], ["h", "t", "ry"]), ("B4", b4, [0.6, -0.8], ["ry", "z"])]:
    print(name, "fidelity =", round(check_puzzle(qc, tgt, al), 6), " kapılar:", dict(qc.count_ops()))
b2.draw("mpl")

---
## G · Devre metrikleri ve `transpile`
| Metrik | Anlamı | Qiskit |
|---|---|---|
| Kapı sayısı (size) | Toplam işlem sayısı | `qc.size()` |
| Kapı türleri | Hangi kapıdan kaç tane | `qc.count_ops()` |
| Derinlik (depth) | Paralel katman sayısı = en uzun yol; çalışma süresinin göstergesi | `qc.depth()` |
| Genişlik (width) | Kübit (+ klasik bit) sayısı | `qc.num_qubits`, `qc.width()` |

Gerçek donanım sadece birkaç **temel kapıyı** çalıştırır. IBM tarzı hedef set: `['rz', 'sx', 'x', 'cx']`. `transpile`, devreyi bu sete çevirir ve **optimize eder** (gcc -O0…-O3 benzetmesi).

In [ ]:
qc1 = QuantumCircuit(1)
qc1.h(0); qc1.t(0); qc1.h(0); qc1.s(0); qc1.rx(0.3, 0); qc1.ry(0.5, 0); qc1.z(0); qc1.h(0)
print("orijinal:", qc1.size(), "kapı, derinlik", qc1.depth(), dict(qc1.count_ops()))
BASIS = ["rz", "sx", "x", "cx"]
for lvl in [0, 3]:
    t = transpile(qc1, basis_gates=BASIS, optimization_level=lvl, seed_transpiler=7)
    print(f"seviye {lvl}: {t.size()} kapı, derinlik {t.depth()}, {dict(t.count_ops())}, eşdeğer mi? {Operator(t).equiv(Operator(qc1))}")
qc1.draw("mpl")

In [ ]:
t3 = transpile(qc1, basis_gates=BASIS, optimization_level=3, seed_transpiler=7)
t3.draw("mpl")    # 8 kapılık devre → rz·sx·rz·sx·rz (her tek kübit kapısı en fazla 5 temel kapı!)

In [ ]:
def demo_circuit():
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])
    qc.t(0); qc.t(0)
    qc.cx(0, 1); qc.rz(0.4, 1); qc.rz(0.3, 1); qc.cx(0, 1)
    qc.s(2); qc.h(2); qc.h(2); qc.sdg(2)
    qc.cx(1, 2); qc.ry(0.3, 2); qc.cx(2, 1); qc.rx(0.2, 1); qc.cx(1, 2); qc.rz(0.5, 2); qc.cx(2, 1); qc.cx(1, 2)
    qc.x(0); qc.ry(0.5, 0); qc.x(0)
    qc.h(1); qc.z(1); qc.h(1)
    qc.cx(0, 2); qc.t(2); qc.cx(0, 2)
    qc.h([0, 1, 2])
    return qc

qd = demo_circuit()
print(f"orijinal: genişlik {qd.num_qubits}, {qd.size()} kapı, derinlik {qd.depth()}")
rows = []
for lvl in range(4):
    t = transpile(qd, basis_gates=BASIS, optimization_level=lvl, seed_transpiler=7)
    rows.append((lvl, t.depth(), t.size(), t.count_ops().get("cx", 0), Operator(t).equiv(Operator(qd))))
print("\nseviye  derinlik  kapı  cx  eşdeğer")
for r in rows: print(f"{r[0]:6d}  {r[1]:8d}  {r[2]:4d}  {r[3]:2d}  {r[4]}")

lv = np.arange(4); w = 0.27
plt.figure(figsize=(8, 3.6))
plt.bar(lv - w, [r[1] for r in rows], w, color=NAVY, label="derinlik")
plt.bar(lv, [r[2] for r in rows], w, color=BLUE, label="kapı sayısı")
plt.bar(lv + w, [r[3] for r in rows], w, color=ORANGE, label="CX")
plt.xticks(lv, [f"seviye {i}" for i in lv]); plt.legend(frameon=False); plt.title("optimization_level karşılaştırması", color=NAVY); plt.show()
qd.draw("mpl", fold=30)

**Gözlemler:** Seviye 0 sadece çevirir (her H → rz·sx·rz gibi), hiç sadeleştirmez. Seviye 1 art arda gelen tek kübit kapılarını birleştirir ve birbirini götüren CX çiftlerini siler. Seviye 2–3 daha pahalı analizlerle (değişme kuralları, iki kübitlik blokların yeniden sentezi) CX sayısını da azaltır. Tüm seviyelerde devre **eşdeğerdir** — tıpkı `gcc -O3`'ün programın anlamını değiştirmemesi gibi.

---
## H · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Faz kapısı fabrikası
`phase_gate(lam)` P(λ) matrisini döndürsün. Qiskit'in `p` kapısıyla karşılaştırın ve T⁴ = Z, S·T = P(3π/4) eşitliklerini doğrulayın.

In [ ]:
def phase_gate(lam):
    # TODO: 2x2 karmaşık matris
    pass

for lam in [0.1, np.pi/4, 2.0]:
    qc = QuantumCircuit(1); qc.p(lam, 0)
    assert np.allclose(phase_gate(lam), Operator(qc).data)
Tm = phase_gate(np.pi/4)
assert np.allclose(np.linalg.matrix_power(Tm, 4), Z)
assert np.allclose(phase_gate(np.pi/2) @ Tm, phase_gate(3*np.pi/4))
print("Alıştırma 1 ✓")

### Alıştırma 2 · Genel döndürme fonksiyonu
`rot(axis, theta)`: `axis` ∈ {"x","y","z"} için Rx/Ry/Rz matrisini döndürsün. İpucu: R_n(θ) = cos(θ/2)·I − i·sin(θ/2)·σ, burada σ ∈ {X, Y, Z}.

In [ ]:
def rot(axis, theta):
    # TODO
    pass

for ax_ in "xyz":
    for th_ in [0.3, np.pi/2, 2.5]:
        qc = QuantumCircuit(1); getattr(qc, "r" + ax_)(th_, 0)
        assert np.allclose(rot(ax_, th_), Operator(qc).data), (ax_, th_)
print("Alıştırma 2 ✓")

### Alıştırma 3 · İstenen olasılık için θ
`theta_for_prob(p)`: Ry(θ)|0⟩ ölçüldüğünde P(1) = p olacak θ ∈ [0, π] değerini döndürsün. (P(1) = sin²(θ/2) denklemini θ için çözün.)

In [ ]:
def theta_for_prob(p):
    # TODO
    pass

for p in [0.0, 0.1, 0.25, 0.5, 0.9, 1.0]:
    qc = QuantumCircuit(1); qc.ry(theta_for_prob(p), 0)
    assert np.isclose(Statevector(qc).probabilities()[1], p), p
print("θ(0.25) =", theta_for_prob(0.25), "= π/3 ?", np.isclose(theta_for_prob(0.25), np.pi/3))
print("Alıştırma 3 ✓")

### Alıştırma 4 · Her kapı bir U'dur
`as_u_circuit(M)`: 2×2 üniter M için `OneQubitEulerDecomposer("U")` ile (θ, φ, λ) açılarını bulup **tek bir `u` kapılı** devre döndürsün. H·T·H·S dizisinin tek U'ya indiğini gösterin.

In [ ]:
def as_u_circuit(M):
    # TODO
    pass

seq = QuantumCircuit(1); seq.h(0); seq.t(0); seq.h(0); seq.s(0)
M = Operator(seq).data
qu = as_u_circuit(M)
assert qu.size() == 1 and "u" in qu.count_ops()
assert Operator(qu).equiv(Operator(seq))
for s_ in range(5):
    Rm = random_unitary(2, seed=s_).data
    assert Operator(as_u_circuit(Rm)).equiv(Operator(Rm))
print("Alıştırma 4 ✓", qu.data[0].operation.params)

### Alıştırma 5 · `prepare_rz`: sadece Ry ve Rz ile hazırlama
Bölüm F'deki `prepare` P kapısı kullanıyordu. Bu kez **yalnız `ry` ve `rz`** kullanan `prepare_rz(alpha, beta)` yazın. 100 rastgele hedefte fidelity ≥ 0.999 olmalı. (Neden P yerine Rz kullanabiliyoruz?)

In [ ]:
def prepare_rz(alpha, beta):
    # TODO
    pass

for _ in range(100):
    v = rng.normal(size=2) + 1j*rng.normal(size=2); v /= np.linalg.norm(v)
    qc = prepare_rz(*v)
    assert set(qc.count_ops()) <= {"ry", "rz"}
    assert state_fidelity(Statevector(qc), Statevector(v)) >= 0.999
print("Alıştırma 5 ✓")

### Alıştırma 6 · Hedef duruma ulaş (4 bulmaca)
`check_puzzle` fonksiyonunu kullanarak dört devre kurun:
- **P1:** hedef |−⟩, kapılar {H, S}
- **P2:** hedef [cos(π/8), −i·sin(π/8)], kapılar {H, T} (ipucu: H·Z·H = X ise H·T·H = ?)
- **P3:** hedef |+i⟩, kapılar {Rx} (tek kapı)
- **P4:** hedef [0.8, 0.6i], kapılar {Ry, S}

In [ ]:
p1 = QuantumCircuit(1)   # TODO
p2 = QuantumCircuit(1)   # TODO
p3 = QuantumCircuit(1)   # TODO
p4 = QuantumCircuit(1)   # TODO

check_puzzle(p1, np.array([1, -1])/np.sqrt(2), ["h", "s"])
check_puzzle(p2, [np.cos(np.pi/8), -1j*np.sin(np.pi/8)], ["h", "t"])
check_puzzle(p3, np.array([1, 1j])/np.sqrt(2), ["rx"]); assert p3.size() == 1
check_puzzle(p4, [0.8, 0.6j], ["ry", "s"])
print("Alıştırma 6 ✓")

### Alıştırma 7 · Elle sadeleştirme
Aşağıdaki `long` devresine **eşdeğer** ama en fazla **2 kapılık** bir `short` devresi yazın (transpile kullanmadan; birleştirme kurallarını düşünün). `metrics(qc)` fonksiyonu (size, depth, width) üçlüsünü döndürsün.

In [ ]:
long = QuantumCircuit(1)
long.t(0); long.t(0); long.s(0); long.rz(0.2, 0); long.rz(-0.2, 0); long.h(0); long.x(0); long.x(0)

def metrics(qc):
    # TODO: (size, depth, width)
    pass

short = QuantumCircuit(1)   # TODO

assert metrics(long) == (8, 8, 1)
assert Operator(short).equiv(Operator(long)) and metrics(short)[0] <= 2
print("Alıştırma 7 ✓", metrics(long), "→", metrics(short))

### Alıştırma 8 · Transpile deney tablosu
`transpile_table(qc)` her optimization_level (0–3) için `(seviye, derinlik, kapı sayısı, cx sayısı)` dörtlülerinden oluşan liste döndürsün (hedef set `BASIS`, `seed_transpiler=7`). `demo_circuit()` üzerinde çalıştırın; seviye 3'ün seviye 0'dan daha az kapı kullandığını ve tüm sonuçların orijinale eşdeğer olduğunu doğrulayın.

In [ ]:
def transpile_table(qc):
    # TODO
    pass

tab = transpile_table(demo_circuit())
for r in tab: print(r)
assert len(tab) == 4 and tab[3][2] < tab[0][2] and tab[3][3] <= tab[0][3]
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- Faz kapıları (S, S†, T, T†, P(λ)) = z ekseni etrafında dönüş; olasılıkları değiştirmez, göreli fazı değiştirir. T² = S, S² = Z.
- Rx, Ry, Rz: eksen etrafında θ kadar dönüş; matrislerde θ/2 çünkü Bloch açısı genlik açısının iki katıdır. Ry(θ)|0⟩ için P(1) = sin²(θ/2).
- Rz(λ) ile P(λ) sadece global faz kadar farklıdır: aynı işlem.
- Her tek kübit kapısı bir U(θ, φ, λ)'dır; Euler: U = Rz·Ry·Rz (global faz hariç).
- Aynı eksende açılar toplanır; farklı eksenler değişmeli değildir.
- `prepare(α, β)`: θ = 2·arccos|α| (ry), φ = arg β − arg α (p).
- Metrikler: size, depth, width; `transpile` = kuantum derleyicisi, seviye 0–3 ≈ gcc -O0…-O3.

**Gelecek hafta:** Çok kübitli devreler: CNOT, CZ, SWAP, Toffoli; dolanıklık, Bell ve GHZ durumları, tersinir klasik mantık (yarım toplayıcı) ve kuantum ışınlanma devresi.